# D1S Decay-Photon Transport - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC D1S (Direct 1-Step) decay-photon transport results using a broomstick (cylinder) geometry with the ENDF/B-VIII.1 nuclear data library.

## Model Description

- **Geometry**: Cylinder with radius 1 cm, height 100 cm (z = -50 to +50), vacuum boundaries
- **Source**: Point source at origin, 14.06 MeV monoenergetic neutrons, isotropic
- **Material**: Single nuclide at 1.0 g/cm³, temperature 294 K
- **Transport**: Coupled neutron-photon with decay photons (`use_decay_photons=True`)
- **Statistics**: 500,000 particles × 10 batches = 5,000,000 total histories
- **Timesteps**: 3 × 1 hour irradiation + 2 × 15 day cooling
- **Tallies**: TCF-corrected photon flux spectra (VITAMIN-J-42) at each timestep

In [1]:
# Setup and imports
import re
import time
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import glob
import os
import yamc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
PARTICLES = 500000
BATCHES = 10
SOURCE_ENERGY = 14.06e6  # 14.06 MeV neutrons
FLUX_DECADES = 6

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None

# Cylinder dimensions
CYLINDER_RADIUS = 1.0  # cm
CYLINDER_HALF_HEIGHT = 50.0  # cm

# Group structure for photon spectra
PHOTON_GROUP_STRUCTURE = "VITAMIN-J-42"

# Chain file
CHAIN_FILE = "/home/jon/yamc-org/yamc/crates/yamc/tests/chain_endf_b8.1_sfr.xml"

# D1S parameters
D1S_TIMESTEPS = [3600.0, 3600.0, 3600.0, 15 * 86400.0, 15 * 86400.0]
D1S_SOURCE_RATES = [1.0, 1.0, 1.0, 0.0, 0.0]
D1S_PHASES = ["irrad", "irrad", "irrad", "cool", "cool"]

In [2]:
def run_openmc_d1s(nuclide, neutron_path, element, photon_path):
    """Run OpenMC D1S simulation with broomstick geometry.

    Returns dict with TCF-corrected photon flux spectra at each timestep.
    """
    import openmc
    from openmc.deplete import d1s

    openmc.reset_auto_ids()
    openmc.config['cross_sections'] = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'cross_sections.xml')
    openmc.config['chain_file'] = CHAIN_FILE

    material = openmc.Material()
    material.temperature = 294
    material.add_nuclide(nuclide, 1.0)
    material.set_density('g/cm3', 1.0)

    cylinder = openmc.ZCylinder(r=CYLINDER_RADIUS, boundary_type='vacuum')
    z_bot = openmc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    z_top = openmc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    region = -cylinder & +z_bot & -z_top

    cell = openmc.Cell(name='broomstick', region=region, fill=material)
    geometry = openmc.Geometry([cell])

    source = openmc.IndependentSource(
        space=openmc.stats.Point((0, 0, 0)),
        angle=openmc.stats.Isotropic(),
        energy=openmc.stats.Discrete([SOURCE_ENERGY], [1]),
    )

    settings = openmc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False},
    )
    settings.photon_transport = True
    settings.use_decay_photons = True

    cell_filter = openmc.CellFilter(cell)
    photon_filter = openmc.ParticleFilter(['photon'])
    energy_filter = openmc.EnergyFilter.from_group_structure(PHOTON_GROUP_STRUCTURE)
    energy_bins_2d = energy_filter.bins
    energy_bins = np.concatenate([[energy_bins_2d[0, 0]], energy_bins_2d[:, 1]])

    tally = openmc.Tally(name='d1s_spectrum')
    tally.filters = [cell_filter, photon_filter, energy_filter]
    tally.scores = ['flux']

    model = openmc.Model(
        geometry, openmc.Materials([material]), settings,
        openmc.Tallies([tally]),
    )

    d1s.prepare_tallies(model=model)
    radionuclides = d1s.get_radionuclides(model)

    t0 = time.time()
    sp_path = model.run()
    elapsed = time.time() - t0

    with openmc.StatePoint(sp_path) as sp:
        tally_result = sp.get_tally(name='d1s_spectrum')

    tcf = d1s.time_correction_factors(
        nuclides=radionuclides, timesteps=D1S_TIMESTEPS,
        source_rates=D1S_SOURCE_RATES, timestep_units='s',
    )

    spectra = []
    spectra_std = []
    for step_idx in range(len(D1S_TIMESTEPS)):
        corrected = d1s.apply_time_correction(
            tally=tally_result, time_correction_factors=tcf,
            index=step_idx + 1, sum_nuclides=True,
        )
        spectra.append(corrected.mean.flatten().tolist())
        spectra_std.append(corrected.std_dev.flatten().tolist())

    return {
        'radionuclides': radionuclides,
        'energy_bins': energy_bins,
        'spectra': spectra,
        'spectra_std': spectra_std,
        'elapsed': elapsed,
    }

In [3]:
def run_yamc_d1s(nuclide, neutron_path, element, photon_path):
    """Run YAMC D1S simulation with broomstick geometry. Returns (results_dict, model)."""
    import yamc as mc
    from yamc import d1s

    material = mc.Material(
        composition={nuclide: 1.0},
        density=1.0,
        temperature=294,
        id=1,
    )
    material.read_nuclear_data('endf-b8.1')

    cylinder = mc.ZCylinder(radius=CYLINDER_RADIUS, boundary='vacuum')
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary='vacuum')
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary='vacuum')
    region = -cylinder & +z_bot & -z_top

    cell = mc.Cell(id=1, name='broomstick', region=region, material=material)
    geometry = mc.Geometry([cell])

    source = mc.NeutronSource(
        position=mc.PointSource(0, 0, 0),
        direction=mc.Isotropic(),
        energy=mc.Discrete([SOURCE_ENERGY], [1.0]),
    )

    # Chain file is set via module-level attribute for D1S
    mc.transmutation_data = CHAIN_FILE

    radionuclides = d1s.get_radionuclides_from_chain(
        nuclide_names=[nuclide], transmutation_file=CHAIN_FILE
    )

    tally = mc.Tally(
        scores=['flux'],
        name='d1s_spectrum',
        cell=cell,
        particle='photon',
        energy_group_structure=PHOTON_GROUP_STRUCTURE,
        parent_nuclides=radionuclides,
    )

    energy_bins = np.array(tally.energy_bins)

    model = mc.Model(
        geometry=geometry,
        tallies=[tally],
        source=source,
        particles_per_batch=PARTICLES,
        batches=BATCHES,
        seed=1,
        photon_transport=True,
        use_decay_photons=True,
    )

    t0 = time.time()
    sim_results = model.simulate()
    elapsed = time.time() - t0

    tcf = d1s.time_correction_factors(
        nuclides=radionuclides, timesteps=D1S_TIMESTEPS,
        source_rates=D1S_SOURCE_RATES, transmutation_file=CHAIN_FILE,
    )

    spectra = []
    spectra_std = []
    for step_idx in range(len(D1S_TIMESTEPS)):
        corrected = d1s.apply_time_correction(
            tally=tally, tcf=tcf, index=step_idx + 1, sum_nuclides=True,
        )
        spectra.append(list(corrected['mean']))
        spectra_std.append(list(corrected['standard_deviation']))

    return {
        'radionuclides': radionuclides,
        'energy_bins': energy_bins,
        'spectra': spectra,
        'spectra_std': spectra_std,
        'elapsed': elapsed,
    }, model

In [4]:
def create_step_trace(energy_bins, values, name, color, dash=None):
    """Create a step plot trace for Plotly."""
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])

    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
    )


def create_comparison_plot(energy_bins, openmc_flux, openmc_std, yamc_flux, yamc_std, title="Flux"):
    """Create a Plotly subplot with photon flux spectra (top) and ratio (bottom)."""

    energy_bins = np.asarray(energy_bins)
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])

    omc_flux = np.asarray(openmc_flux, dtype=float).copy()
    omc_std = np.asarray(openmc_std, dtype=float).copy()
    ymc_flux = np.asarray(yamc_flux, dtype=float).copy()
    ymc_std = np.asarray(yamc_std, dtype=float).copy()

    # Normalize by lethargy bin width
    lethargy_bin_width = np.log10(energy_bins[1:] / energy_bins[:-1])
    omc_flux /= lethargy_bin_width
    omc_std /= lethargy_bin_width
    ymc_flux /= lethargy_bin_width
    ymc_std /= lethargy_bin_width

    has_flux = (omc_flux > 0) | (ymc_flux > 0)
    if np.any(has_flux):
        first_nonzero_idx = np.argmax(has_flux)
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]

    max_plot_energy = energy_bins[-1] * 1.1

    max_flux = max(
        np.max(omc_flux[omc_flux > 0]) if np.any(omc_flux > 0) else 1e-10,
        np.max(ymc_flux[ymc_flux > 0]) if np.any(ymc_flux > 0) else 1e-10
    )
    flux_threshold = max_flux / (10 ** FLUX_DECADES)

    omc_flux_plot = np.where(omc_flux >= flux_threshold, omc_flux, np.nan)
    ymc_flux_plot = np.where(ymc_flux >= flux_threshold, ymc_flux, np.nan)
    omc_std_plot = np.where(omc_flux >= flux_threshold, omc_std, np.nan)
    ymc_std_plot = np.where(ymc_flux >= flux_threshold, ymc_std, np.nan)

    valid = (omc_flux > 0) & (ymc_flux > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(valid, ymc_flux / omc_flux, np.nan)
        rel_err_yamc = np.where(ymc_flux > 0, ymc_std / ymc_flux, 0)
        rel_err_openmc = np.where(omc_flux > 0, omc_std / omc_flux, 0)
    ratio_rel_err = np.sqrt(rel_err_yamc**2 + rel_err_openmc**2)
    ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=('Photon Flux per Unit Lethargy', 'YAMC / OpenMC Ratio')
    )

    fig.add_trace(
        create_step_trace(energy_bins, omc_flux_plot, 'OpenMC', 'blue'),
        row=1, col=1
    )
    fig.add_trace(
        create_step_trace(energy_bins, ymc_flux_plot, 'YAMC', 'red', dash='dash'),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=omc_flux_plot, mode='markers',
            marker=dict(color='blue', size=4, symbol='circle'),
            error_y=dict(type='data', array=omc_std_plot, visible=True, color='blue', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=ymc_flux_plot, mode='markers',
            marker=dict(color='red', size=4, symbol='square'),
            error_y=dict(type='data', array=ymc_std_plot, visible=True, color='red', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )

    fig.add_trace(
        create_step_trace(energy_bins, ratio, 'YAMC/OpenMC', 'green'),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([bin_centers, bin_centers[::-1]]),
            y=np.concatenate([ratio + ratio_std, (ratio - ratio_std)[::-1]]),
            fill='toself',
            fillcolor='rgba(0, 128, 0, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            name='Uncertainty',
            showlegend=False,
        ),
        row=2, col=1
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

    fig.update_xaxes(type="log", title_text="Energy (eV)",
                     range=[np.log10(min_plot_energy), np.log10(max_plot_energy)],
                     exponentformat='e', row=2, col=1)
    fig.update_xaxes(type="log",
                     range=[np.log10(min_plot_energy), np.log10(max_plot_energy)],
                     exponentformat='e', row=1, col=1)
    fig.update_yaxes(type="log", title_text="Photon Flux / Lethargy", exponentformat='e', row=1, col=1)
    fig.update_yaxes(title_text="Ratio", range=[0.5, 1.5], row=2, col=1)

    fig.update_layout(
        height=700,
        title=title,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=60, r=20, t=60, b=40)
    )

    return fig


def d1s_quality_label(max_rel_diff):
    """Assign quality label based on max relative difference (%) for D1S."""
    if max_rel_diff < 2.0:
        return "Excellent"
    elif max_rel_diff < 5.0:
        return "Good"
    elif max_rel_diff < 10.0:
        return "Fair"
    else:
        return "Poor"


def rel_diff(a, b):
    """Safe relative difference: (b - a) / a * 100, returns NaN if a == 0."""
    if a == 0:
        return np.nan
    return (b - a) / a * 100


def compute_step_metrics(openmc_result, yamc_result):
    """Compute per-timestep comparison metrics."""
    step_metrics = []
    for step_idx in range(len(D1S_TIMESTEPS)):
        omc_spec = np.array(openmc_result['spectra'][step_idx])
        ymc_spec = np.array(yamc_result['spectra'][step_idx])

        omc_total = omc_spec.sum()
        ymc_total = ymc_spec.sum()
        total_rel = rel_diff(omc_total, ymc_total)

        step_metrics.append({
            'step': step_idx + 1,
            'phase': D1S_PHASES[step_idx],
            'omc_total': float(omc_total),
            'ymc_total': float(ymc_total),
            'total_rel_pct': float(total_rel),
        })

    finite_diffs = [
        abs(m['total_rel_pct']) for m in step_metrics
        if np.isfinite(m['total_rel_pct'])
    ]
    max_total_rel = max(finite_diffs) if finite_diffs else 0.0
    quality = d1s_quality_label(max_total_rel)
    speedup = openmc_result['elapsed'] / yamc_result['elapsed'] if yamc_result['elapsed'] > 0 else float('inf')

    return {
        'step_metrics': step_metrics,
        'max_total_rel_pct': max_total_rel,
        'quality': quality,
        'speedup': speedup,
    }


def create_step_metrics_table(metrics):
    """Create a table of per-timestep comparison metrics."""
    rows = []
    for m in metrics['step_metrics']:
        rows.append({
            'Step': m['step'],
            'Phase': m['phase'],
            'OpenMC Total': f"{m['omc_total']:.4e}",
            'YAMC Total': f"{m['ymc_total']:.4e}",
            'Rel. Diff (%)': f"{m['total_rel_pct']:+.2f}%" if np.isfinite(m['total_rel_pct']) else 'N/A',
        })

    rows.append({
        'Step': '',
        'Phase': '',
        'OpenMC Total': '',
        'YAMC Total': 'Quality:',
        'Rel. Diff (%)': metrics['quality'],
    })

    return pd.DataFrame(rows)

In [5]:
# Discover D1S-relevant nuclides: those that produce radionuclides with decay photon sources
def element_from_nuclide(nuclide):
    """Extract element symbol from nuclide name: 'Fe56' -> 'Fe'."""
    return re.sub(r"\d+", "", nuclide)


def find_d1s_nuclides():
    """Find nuclides that produce radionuclides with decay photon sources."""
    tree = ET.parse(CHAIN_FILE)
    root = tree.getroot()

    half_lives = {}
    reactions_map = {}
    photon_sources = set()

    for nuc_elem in root.findall('nuclide'):
        name = nuc_elem.get('name')
        hl = nuc_elem.get('half_life')
        if hl is not None:
            half_lives[name] = float(hl)
        for source in nuc_elem.findall('source'):
            if source.get('particle') == 'photon':
                photon_sources.add(name)
        for rx in nuc_elem.findall('reaction'):
            target = rx.get('target')
            if name not in reactions_map:
                reactions_map[name] = []
            reactions_map[name].append(target)

    d1s_parents = set()
    for parent, targets in reactions_map.items():
        for target in targets:
            if target and target != 'None':
                if target in half_lives and target in photon_sources:
                    d1s_parents.add(parent)

    neutron_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron')
    photon_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'photon')
    h5_files = sorted(glob.glob(os.path.join(neutron_dir, '*.h5')))

    result = []
    for f in h5_files:
        name = os.path.splitext(os.path.basename(f))[0]
        if '_m' in name:
            continue
        if name not in d1s_parents:
            continue
        element = element_from_nuclide(name)
        photon_path = os.path.join(photon_dir, f"{element}.h5")
        if not os.path.exists(photon_path):
            continue
        result.append((name, f, element, photon_path))

    return result


nuclides = find_d1s_nuclides()

if not nuclides:
    raise FileNotFoundError(
        f"No D1S-relevant nuclides found.\n"
        f"Please check that the chain file and nuclear data directories exist."
    )

if NUMBER_OF_SIMULATIONS:
    nuclides = nuclides[:NUMBER_OF_SIMULATIONS]

print(f"Running D1S simulations for {len(nuclides)} nuclides...")
for nuc, _, elem, _ in nuclides:
    print(f"  {nuc} ({elem})")

Running D1S simulations for 3 nuclides...
  Ac225 (Ac)
  Ac226 (Ac)
  Ac227 (Ac)


In [6]:
# Run all simulations and collect results
all_results = {}
first_yamc_model = None

for nuclide, neutron_path, element, photon_path in nuclides:
    print(f"\nProcessing {nuclide} ({element})...")

    try:
        print("  Running YAMC D1S...")
        yamc_result, yamc_model = run_yamc_d1s(nuclide, neutron_path, element, photon_path)
        if first_yamc_model is None:
            first_yamc_model = yamc_model
        print(f"    {yamc_result['elapsed']:.1f}s, {len(yamc_result['radionuclides'])} radionuclides")
    except Exception as e:
        print(f"  YAMC Error: {e}")
        continue

    try:
        print("  Running OpenMC D1S...")
        openmc_result = run_openmc_d1s(nuclide, neutron_path, element, photon_path)
        print(f"    {openmc_result['elapsed']:.1f}s, {len(openmc_result['radionuclides'])} radionuclides")
    except Exception as e:
        print(f"  OpenMC Error: {e}")
        continue

    metrics = compute_step_metrics(openmc_result, yamc_result)
    all_results[nuclide] = {
        'openmc': openmc_result,
        'yamc': yamc_result,
        'metrics': metrics,
        'element': element,
    }

    for m in metrics['step_metrics']:
        print(f"    Step {m['step']} ({m['phase']:>5}): "
              f"OpenMC={m['omc_total']:.4e}  YAMC={m['ymc_total']:.4e}  "
              f"diff={m['total_rel_pct']:+.2f}%")
    print(f"  Quality: {metrics['quality']} | Speedup: {metrics['speedup']:.2f}x")

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")


Processing Ac225 (Ac)...
  Running YAMC D1S...
  YAMC Error: Failed to load chain: chain path '/home/jon/yamc-org/yamc/crates/yamc/tests/chain_endf_b8.1_sfr.xml' is not a directory; expected a `.chain.arrow/` layout

Processing Ac226 (Ac)...
  Running YAMC D1S...
  YAMC Error: Failed to load chain: chain path '/home/jon/yamc-org/yamc/crates/yamc/tests/chain_endf_b8.1_sfr.xml' is not a directory; expected a `.chain.arrow/` layout

Processing Ac227 (Ac)...
  Running YAMC D1S...
  YAMC Error: Failed to load chain: chain path '/home/jon/yamc-org/yamc/crates/yamc/tests/chain_endf_b8.1_sfr.xml' is not a directory; expected a `.chain.arrow/` layout

Completed 0 / 3 nuclides successfully.


In [7]:
# Display broomstick model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", n_samples=500, axis_units="cm")
    html_plot.save("d1s_model_plot.html")
    encoded = base64.b64encode(html_plot.html.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

No successful YAMC simulation to plot.


## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- TCF-corrected photon flux spectrum plot per timestep (with lethargy normalization)
- Per-timestep comparison metrics table
- Quality assessment
- Performance comparison

In [8]:
# Display results for each nuclide
for nuclide, results in all_results.items():
    openmc_result = results['openmc']
    yamc_result = results['yamc']
    metrics = results['metrics']
    element = results['element']

    # Header
    display(Markdown(f"## {nuclide} ({element})"))

    # Performance comparison
    omc_time = openmc_result['elapsed']
    ymc_time = yamc_result['elapsed']
    display(Markdown(
        f"**Performance:** OpenMC: {omc_time:.1f}s | YAMC: {ymc_time:.1f}s | Speedup: {metrics['speedup']:.2f}x  \n"
        f"**Radionuclides:** OpenMC: {len(openmc_result['radionuclides'])} | YAMC: {len(yamc_result['radionuclides'])}"
    ))

    # Step metrics table
    display(Markdown("### Step Comparison"))
    metrics_table = create_step_metrics_table(metrics)
    display(metrics_table.style.hide(axis='index'))

    # Spectrum plot per timestep
    energy_bins = openmc_result['energy_bins']

    for step_idx in range(len(D1S_TIMESTEPS)):
        phase = D1S_PHASES[step_idx]
        display(Markdown(f"### Step {step_idx + 1} ({phase})"))

        fig = create_comparison_plot(
            energy_bins,
            np.array(openmc_result['spectra'][step_idx]),
            np.array(openmc_result['spectra_std'][step_idx]),
            np.array(yamc_result['spectra'][step_idx]),
            np.array(yamc_result['spectra_std'][step_idx]),
            title=f"{nuclide} - Step {step_idx + 1} ({phase}) - Decay Photon Spectrum",
        )
        fig.show()

    print()  # Add spacing between nuclides